# Example 1.1.1 / 3: From an Indexed Set to a Set of Pairs

Companion notebook for **1.1.1 Flat vs Indexed Sets** in the AMPL lab Documentation, section "From an Indexed Set to a Set of Pairs". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## From an Indexed Set to a Set of Pairs

`setof` builds a flat set from an indexing expression. It turns an indexed family into one set of
pairs. Here `NEXT[i]` lists the nodes reachable from `i` in this small network, and each arrow
becomes one pair of `ALL_ARCS`:


The model and its data:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES;
set NEXT {NODES} within NODES;     # NEXT[i]: the nodes reachable from i
set ALL_ARCS = setof {i in NODES, j in NEXT[i]} (i,j);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["NODES"] = ["a", "b", "c", "d"]
ampl.set["NEXT"] = {"a": ["b", "c"], "b": ["d"], "c": ["d"], "d": []}

In [5]:
# run.py
print(ampl.set["ALL_ARCS"].to_list())

[('a', 'b'), ('a', 'c'), ('b', 'd'), ('c', 'd')]


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. `set NEXT[d] := ;` gives `NEXT["d"]` no members,
like the empty list in `data.py`:

In [6]:
%%writefile data.dat
set NODES := a b c d;
set NEXT[a] := b c;
set NEXT[b] := d;
set NEXT[c] := d;
set NEXT[d] := ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display ALL_ARCS;

set ALL_ARCS := (a,b) (a,c) (b,d) (c,d);

